In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
KILLS = 0.3
ASSISTS = 0.15
DEATHS_FLAT = 3
DEATHS_MULT = -0.3
LAST_HITS = 0.003
GPM = 0.002
ROSHAN = 1
TOWERS = 1
TEAMFIGHTS = 3
FB = 4
STUNS = 0.05
RUNES = 0.25
WARDS = 0.5
STACKS = 0.5

In [3]:
match_df = pd.read_csv('player_data.csv')

C:\Users\wicki\Anaconda3\envs\data-science\lib\site-packages\IPython\core\interactiveshell.py:3165: DtypeWarning: Columns (17) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [4]:
match_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103480 entries, 0 to 103479
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   match_id                 103480 non-null  int64  
 1   account_id               103480 non-null  int64  
 2   assists                  103480 non-null  int64  
 3   camps_stacked            99774 non-null   float64
 4   deaths                   103480 non-null  int64  
 5   denies                   103480 non-null  int64  
 6   firstblood_claimed       98801 non-null   float64
 7   gold_per_min             103480 non-null  int64  
 8   kills                    103480 non-null  int64  
 9   last_hits                103480 non-null  int64  
 10  obs_placed               99774 non-null   float64
 11  roshans_killed           99161 non-null   float64
 12  rune_pickups             99774 non-null   float64
 13  stuns                    100258 non-null  float64
 14  team

In [61]:
combined_df['team'] = np.where(combined_df['isRadiant'],combined_df['radiant_team_id'],combined_df['dire_team_id'])
combined_df['opp_team'] = np.where(combined_df['isRadiant'],combined_df['dire_team_id'],combined_df['radiant_team_id'])

In [62]:
combined_df.groupby(['series_id','account_id']).head(2)

,match_id,account_id,assists,camps_stacked,deaths,denies,firstblood_claimed,gold_per_min,kills,last_hits,...,win,lose,duration,dire_team_id,radiant_team_id,leagueid,series_id,series_type,team,opp_team
0,6649226879,86745912,8,5.0,0,20,0.0,554,4,170,...,1,0,1213,8261882,39,14281,682742,1,39,8261882
1,6649226879,154715080,10,2.0,2,8,0.0,482,9,122,...,1,0,1213,8261882,39,14281,682742,1,39,8261882
2,6649226879,124801257,17,0.0,1,13,0.0,496,2,121,...,1,0,1213,8261882,39,14281,682742,1,39,8261882
3,6649226879,94155156,9,0.0,3,1,0.0,256,1,24,...,1,0,1213,8261882,39,14281,682742,1,39,8261882
4,6649226879,25907144,13,2.0,2,1,1.0,316,8,11,...,1,0,1213,8261882,39,14281,682742,1,39,8261882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,6381532661,164217422,12,4.0,3,1,0.0,678,11,464,...,0,1,2448,8261882,8606828,13742,633442,1,8261882,8606828
9996,6381532661,108624725,18,3.0,8,0,0.0,274,4,59,...,0,1,2448,8261882,8606828,13742,633442,1,8261882,8606828
9997,6381532661,93159453,17,0.0,9,0,0.0,333,1,153,...,0,1,2448,8261882,8606828,13742,633442,1,8261882,8606828
9998,6381532661,115750060,15,0.0,10,0,0.0,242,1,40,...,0,1,2448,8261882,8606828,13742,633442,1,8261882,8606828


In [65]:
combined_df['kill_score'] = combined_df['kills'] * KILLS
combined_df['assist_score'] = combined_df['assists'] * ASSISTS
combined_df['death_score'] = (DEATHS_FLAT + combined_df['deaths'] * DEATHS_MULT)
combined_df['lasthit_score'] = (combined_df['last_hits'] + combined_df['denies']) * LAST_HITS
combined_df['gpm_score'] = combined_df['gold_per_min'] * GPM
combined_df['fb_score'] = combined_df['firstblood_claimed'] * FB
combined_df['tower_score'] = combined_df['towers_killed'] * TOWERS
combined_df['stack_score'] = combined_df['camps_stacked'] * STACKS
combined_df['roshan_score'] = combined_df['roshans_killed'] * ROSHAN
combined_df['stun_score'] = combined_df['stuns'] * STUNS
combined_df['teamfight_score'] = combined_df['teamfight_participation'] * TEAMFIGHTS
combined_df['obs_score'] = combined_df['obs_placed'] * WARDS
combined_df['rune_score'] = combined_df['rune_pickups'] * RUNES
combined_df['fantasy_score'] = combined_df['kill_score'] + \
                        combined_df['assist_score'] + \
                        combined_df['death_score'] + \
                        combined_df['lasthit_score'] + \
                        combined_df['gpm_score'] + \
                        combined_df['fb_score'] + \
                        combined_df['tower_score'] + \
                        combined_df['stack_score'] + \
                        combined_df['roshan_score'] + \
                        combined_df['stun_score'] + \
                        combined_df['teamfight_score'] + \
                        combined_df['obs_score'] + \
                        combined_df['rune_score'] 

In [66]:
combined_df.sort_values('fantasy_score',ascending=False,inplace=True)

In [67]:
best_2_df = combined_df.copy().groupby(['series_id','account_id']).head(2)

In [68]:
best_2_df = best_2_df.groupby(['series_id','account_id','team','opp_team','leagueid']).sum().reset_index()

In [69]:
best_2_df.drop(['match_id','win','lose','region','radiant_win','isRadiant','radiant_team_id','dire_team_id','start_time'],axis=1,inplace=True)

In [70]:
best_2_df

,series_id,account_id,team,opp_team,leagueid,assists,camps_stacked,deaths,denies,firstblood_claimed,...,gpm_score,fb_score,tower_score,stack_score,roshan_score,stun_score,teamfight_score,obs_score,rune_score,fantasy_score
0,633442,41637292,8606828,8261882,13742,20,0.0,5,25,0.0,...,0.864,0.0,2.0,0.0,0.0,0.000000,1.578947,0.0,1.25,10.918947
1,633442,92706637,8606828,8261882,13742,11,1.0,0,12,0.0,...,1.546,0.0,1.0,0.5,0.0,1.364791,2.368421,1.0,4.00,23.500212
2,633442,93159453,8261882,8606828,13742,17,0.0,9,0,0.0,...,0.666,0.0,1.0,0.0,1.0,2.193396,1.800000,0.0,0.50,10.768397
3,633442,108624725,8261882,8606828,13742,18,3.0,8,0,0.0,...,0.548,0.0,0.0,1.5,0.0,2.388018,2.200000,3.0,1.00,15.313018
4,633442,115750060,8261882,8606828,13742,15,0.0,10,0,0.0,...,0.484,0.0,0.0,0.0,0.0,3.085634,1.600000,6.5,0.25,14.589634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4680,682742,124801257,39,8261882,14281,30,1.0,5,16,0.0,...,2.104,0.0,6.0,0.5,1.0,2.368873,4.410714,0.0,0.50,29.222587
4681,682742,131777279,8261882,39,14281,8,4.0,14,4,0.0,...,0.896,0.0,1.0,2.0,0.0,0.626908,3.675000,7.0,1.75,21.048908
4682,682742,154715080,39,8261882,14281,21,4.0,3,14,0.0,...,1.924,0.0,0.0,2.0,0.0,1.234314,3.982143,2.5,3.50,28.145457
4683,682742,172739956,8261882,39,14281,7,0.0,5,25,0.0,...,1.626,0.0,0.0,0.0,0.0,2.501165,4.425000,0.5,2.25,19.492165


In [71]:
match_start = combined_df.groupby('series_id').min().reset_index()

In [72]:
match_start = match_start[['series_id','start_time']]

In [73]:
best_2_df = pd.merge(best_2_df,match_start,how='left',on='series_id')

In [74]:
best_2_df['date'] = (best_2_df['start_time']).apply(lambda x: datetime.utcfromtimestamp(x).date())

In [75]:
best_2_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4685 entries, 0 to 4684
Data columns (total 37 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   series_id                4685 non-null   int64  
 1   account_id               4685 non-null   int64  
 2   team                     4685 non-null   int64  
 3   opp_team                 4685 non-null   int64  
 4   leagueid                 4685 non-null   int64  
 5   assists                  4685 non-null   int64  
 6   camps_stacked            4685 non-null   float64
 7   deaths                   4685 non-null   int64  
 8   denies                   4685 non-null   int64  
 9   firstblood_claimed       4685 non-null   float64
 10  gold_per_min             4685 non-null   int64  
 11  kills                    4685 non-null   int64  
 12  last_hits                4685 non-null   int64  
 13  obs_placed               4685 non-null   float64
 14  roshans_killed          

In [76]:
best_2_df.sort_values('series_id',ascending=False).head()

,series_id,account_id,team,opp_team,leagueid,assists,camps_stacked,deaths,denies,firstblood_claimed,...,tower_score,stack_score,roshan_score,stun_score,teamfight_score,obs_score,rune_score,fantasy_score,start_time,date
4684,682742,295877832,8261882,39,14281,6,3.0,8,14,0.0,...,2.0,1.5,0.0,0.879558,2.625000,0.0,0.25,15.082558,1657049409,2022-07-05
4683,682742,172739956,8261882,39,14281,7,0.0,5,25,0.0,...,0.0,0.0,0.0,2.501165,4.425000,0.5,2.25,19.492165,1657049409,2022-07-05
4682,682742,154715080,39,8261882,14281,21,4.0,3,14,0.0,...,0.0,2.0,0.0,1.234314,3.982143,2.5,3.50,28.145457,1657049409,2022-07-05
4681,682742,131777279,8261882,39,14281,8,4.0,14,4,0.0,...,1.0,2.0,0.0,0.626908,3.675000,7.0,1.75,21.048908,1657049409,2022-07-05
4680,682742,124801257,39,8261882,14281,30,1.0,5,16,0.0,...,6.0,0.5,1.0,2.368873,4.410714,0.0,0.50,29.222587,1657049409,2022-07-05


In [77]:
elo_df = pd.read_csv('ELO.csv')

In [78]:
elo_df

,team,date,elo_(k=32)_rating,elo_(k=32)_avg_7d,elo_(k=32)_δ7d,elo_(k=32)_avg_30d,elo_(k=32)_δ30d,elo_(k=64)_rating,elo_(k=64)_avg_7d,elo_(k=64)_δ7d,elo_(k=64)_avg_30d,elo_(k=64)_δ30d,glicko_1_rating,glicko_1_μ,glicko_1_σ,glicko_1_δrat.7d,glicko_2_rating,glicko_2_μ,glicko_2_σ,glicko_2_δrat.7d
0,OG,15-7-2022,1145.78,1149.35,-7.19,1151.75,+17.49,1267.11,1273.66,-13.18,1282.74,+7.32,1921.61,2134.13,85.01,-85.63,1956.03,2076.13,48.04,-2.95
1,Tundra Esports,15-7-2022,1158.21,1168.92,+8.54,1124.08,+46.70,1236.96,1259.57,+13.36,1177.67,+85.27,2007.23,2209.32,80.84,27.24,1950.72,2068.25,47.01,-2.96
2,PSG.LGD,15-7-2022,1187.48,1161.15,+43.06,1152.81,+67.27,1295.43,1230.64,+103.44,1213.00,+137.56,1803.70,2124.61,128.36,-69.95,1942.89,2075.11,52.89,-2.73
3,Team Aster,15-7-2022,1093.18,1100.74,+2.80,1090.68,+17.34,1121.69,1145.63,-5.66,1130.15,+3.34,1941.14,2254.68,125.42,-72.07,1912.46,2036.00,49.41,-2.87
4,Royal Never Give Up,15-7-2022,1093.16,1061.40,+70.73,1065.09,-12.01,1175.19,1091.49,+198.37,1066.02,+45.09,1948.97,2251.20,120.89,-75.64,1903.34,2028.39,50.02,-2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17821,Gorillaz-Pride,28-12-2021,1031.99,1032.78,-1.58,1014.93,+40.41,1057.38,1058.78,-2.83,1030.97,+67.47,1177.53,1605.83,171.32,-49.56,1351.09,1525.33,69.70,-1.96
17822,Inverse,28-12-2021,913.36,911.24,+4.28,954.72,-71.31,859.48,856.04,+6.93,926.77,-121.97,790.74,1257.63,186.76,-45.01,1187.90,1383.47,78.23,-1.75
17823,Electronic Boys,28-12-2021,965.76,965.44,-2.44,965.42,-6.76,978.51,977.70,+3.39,958.27,+21.55,1045.54,1432.26,154.69,-55.73,1137.95,1330.41,76.98,-1.77
17824,felt,28-12-2021,957.97,956.94,+2.07,966.33,-18.55,932.47,930.81,+3.33,950.82,-38.68,901.93,1343.58,176.66,-47.88,1130.79,1312.91,72.85,-1.87


In [79]:
#import requests
#from tqdm import tqdm

In [80]:
#leagues = best_2_df['leagueid'].unique()

In [81]:
#teams_df = pd.DataFrame()
teams_df = pd.read_csv('teams.csv')

#for league in leagues:
    #response = requests.get(f"https://api.opendota.com/api/leagues/{league}/teams")
    #data = response.json()
    #teams_df = pd.concat([teams_df,pd.DataFrame(data)]).reset_index(drop=True)

In [82]:
#teams_df = teams_df.drop_duplicates().reset_index(drop=True)

In [83]:
elo_df

,team,date,elo_(k=32)_rating,elo_(k=32)_avg_7d,elo_(k=32)_δ7d,elo_(k=32)_avg_30d,elo_(k=32)_δ30d,elo_(k=64)_rating,elo_(k=64)_avg_7d,elo_(k=64)_δ7d,elo_(k=64)_avg_30d,elo_(k=64)_δ30d,glicko_1_rating,glicko_1_μ,glicko_1_σ,glicko_1_δrat.7d,glicko_2_rating,glicko_2_μ,glicko_2_σ,glicko_2_δrat.7d
0,OG,15-7-2022,1145.78,1149.35,-7.19,1151.75,+17.49,1267.11,1273.66,-13.18,1282.74,+7.32,1921.61,2134.13,85.01,-85.63,1956.03,2076.13,48.04,-2.95
1,Tundra Esports,15-7-2022,1158.21,1168.92,+8.54,1124.08,+46.70,1236.96,1259.57,+13.36,1177.67,+85.27,2007.23,2209.32,80.84,27.24,1950.72,2068.25,47.01,-2.96
2,PSG.LGD,15-7-2022,1187.48,1161.15,+43.06,1152.81,+67.27,1295.43,1230.64,+103.44,1213.00,+137.56,1803.70,2124.61,128.36,-69.95,1942.89,2075.11,52.89,-2.73
3,Team Aster,15-7-2022,1093.18,1100.74,+2.80,1090.68,+17.34,1121.69,1145.63,-5.66,1130.15,+3.34,1941.14,2254.68,125.42,-72.07,1912.46,2036.00,49.41,-2.87
4,Royal Never Give Up,15-7-2022,1093.16,1061.40,+70.73,1065.09,-12.01,1175.19,1091.49,+198.37,1066.02,+45.09,1948.97,2251.20,120.89,-75.64,1903.34,2028.39,50.02,-2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17821,Gorillaz-Pride,28-12-2021,1031.99,1032.78,-1.58,1014.93,+40.41,1057.38,1058.78,-2.83,1030.97,+67.47,1177.53,1605.83,171.32,-49.56,1351.09,1525.33,69.70,-1.96
17822,Inverse,28-12-2021,913.36,911.24,+4.28,954.72,-71.31,859.48,856.04,+6.93,926.77,-121.97,790.74,1257.63,186.76,-45.01,1187.90,1383.47,78.23,-1.75
17823,Electronic Boys,28-12-2021,965.76,965.44,-2.44,965.42,-6.76,978.51,977.70,+3.39,958.27,+21.55,1045.54,1432.26,154.69,-55.73,1137.95,1330.41,76.98,-1.77
17824,felt,28-12-2021,957.97,956.94,+2.07,966.33,-18.55,932.47,930.81,+3.33,950.82,-38.68,901.93,1343.58,176.66,-47.88,1130.79,1312.91,72.85,-1.87


In [84]:
elo_df['date'] = elo_df['date'].apply(lambda x: datetime.strptime(x,'%d-%m-%Y').date())

In [85]:
elo_df = pd.merge(elo_df,teams_df,how='left',left_on='team',right_on='name')

In [86]:
best_2_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4685 entries, 0 to 4684
Data columns (total 37 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   series_id                4685 non-null   int64  
 1   account_id               4685 non-null   int64  
 2   team                     4685 non-null   int64  
 3   opp_team                 4685 non-null   int64  
 4   leagueid                 4685 non-null   int64  
 5   assists                  4685 non-null   int64  
 6   camps_stacked            4685 non-null   float64
 7   deaths                   4685 non-null   int64  
 8   denies                   4685 non-null   int64  
 9   firstblood_claimed       4685 non-null   float64
 10  gold_per_min             4685 non-null   int64  
 11  kills                    4685 non-null   int64  
 12  last_hits                4685 non-null   int64  
 13  obs_placed               4685 non-null   float64
 14  roshans_killed          

In [87]:
best_2_df = pd.merge(best_2_df,elo_df[['team_id','date','elo_(k=32)_rating']],how='left',left_on=['team','date'],right_on=['team_id','date'])

In [88]:
best_2_df.columns

Index(['series_id', 'account_id', 'team', 'opp_team', 'leagueid', 'assists',
       'camps_stacked', 'deaths', 'denies', 'firstblood_claimed',
       'gold_per_min', 'kills', 'last_hits', 'obs_placed', 'roshans_killed',
       'rune_pickups', 'stuns', 'teamfight_participation', 'towers_killed',
       'duration', 'series_type', 'kill_score', 'assist_score', 'death_score',
       'lasthit_score', 'gpm_score', 'fb_score', 'tower_score', 'stack_score',
       'roshan_score', 'stun_score', 'teamfight_score', 'obs_score',
       'rune_score', 'fantasy_score', 'start_time', 'date', 'team_id',
       'elo_(k=32)_rating'],
      dtype='object')

In [89]:
best_2_df = best_2_df.rename({'elo_(k=32)_rating':'team_elo_(k=32)_rating'})

In [90]:
best_2_df.drop('team_id',axis=1,inplace=True)

In [91]:
best_2_df = pd.merge(best_2_df,elo_df[['team_id','date','elo_(k=32)_rating']],how='left',left_on=['opp_team','date'],right_on=['team_id','date'])

In [92]:
best_2_df.drop('team_id',axis=1,inplace=True)

In [93]:
best_2_df = best_2_df.rename({'elo_(k=32)_rating_x':'team_elo_(k=32)_rating','elo_(k=32)_rating_y':'opp_team_elo_(k=32)_rating'},axis=1)

In [94]:
best_2_df[best_2_df['opp_team_elo_(k=32)_rating'].isna()]['opp_team'].unique()

array([8606828, 8581426, 8607159, 8604954, 8598999, 8271508, 8685387,
       8545488, 8687576, 8687717, 8690622, 8687202, 7300277, 8435977,
       8582076, 8549249, 8590460, 8686363, 8686313, 8494848, 8661276,
       8724882, 8261642, 8644812, 7527649, 8344536, 8726557, 5252228,
       8728920, 8686786, 8724854, 8721219, 8732747, 8741396], dtype=int64)

In [95]:
best_2_df.columns

Index(['series_id', 'account_id', 'team', 'opp_team', 'leagueid', 'assists',
       'camps_stacked', 'deaths', 'denies', 'firstblood_claimed',
       'gold_per_min', 'kills', 'last_hits', 'obs_placed', 'roshans_killed',
       'rune_pickups', 'stuns', 'teamfight_participation', 'towers_killed',
       'duration', 'series_type', 'kill_score', 'assist_score', 'death_score',
       'lasthit_score', 'gpm_score', 'fb_score', 'tower_score', 'stack_score',
       'roshan_score', 'stun_score', 'teamfight_score', 'obs_score',
       'rune_score', 'fantasy_score', 'start_time', 'date',
       'team_elo_(k=32)_rating', 'opp_team_elo_(k=32)_rating'],
      dtype='object')

In [96]:
elo_df[elo_df['team_id']==39]

,team,date,elo_(k=32)_rating,elo_(k=32)_avg_7d,elo_(k=32)_δ7d,elo_(k=32)_avg_30d,elo_(k=32)_δ30d,elo_(k=64)_rating,elo_(k=64)_avg_7d,elo_(k=64)_δ7d,...,glicko_1_μ,glicko_1_σ,glicko_1_δrat.7d,glicko_2_rating,glicko_2_μ,glicko_2_σ,glicko_2_δrat.7d,team_id,name,tag
11,Evil Geniuses,2022-07-15,1136.89,1140.24,-6.75,1138.84,-3.73,1204.28,1209.28,-10.08,...,2070.40,94.55,84.15,1831.90,1966.23,53.73,-2.57,39.0,Evil Geniuses,EG
104,Evil Geniuses,2022-07-14,1136.89,1140.24,-6.75,1138.84,-3.73,1204.28,1209.28,-10.08,...,2070.40,94.55,84.15,1831.90,1966.23,53.73,-2.57,39.0,Evil Geniuses,EG
197,Evil Geniuses,2022-07-13,1136.89,1140.24,-6.75,1138.84,-3.73,1204.28,1209.28,-10.08,...,2070.40,94.55,84.15,1831.90,1966.23,53.73,-2.57,39.0,Evil Geniuses,EG
290,Evil Geniuses,2022-07-12,1136.89,1140.24,-6.75,1138.84,-3.73,1204.28,1209.28,-10.08,...,2070.40,94.55,84.15,1831.90,1966.23,53.73,-2.57,39.0,Evil Geniuses,EG
383,Evil Geniuses,2022-07-11,1136.89,1140.24,-6.75,1138.84,-3.73,1204.28,1209.28,-10.08,...,2070.40,94.55,84.15,1831.90,1966.23,53.73,-2.57,39.0,Evil Geniuses,EG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18433,Evil Geniuses,2022-01-01,1017.40,1017.83,-0.86,1029.03,-41.47,983.27,982.86,+0.83,...,1609.67,181.93,-46.33,1820.31,1965.99,58.27,-2.36,39.0,Evil Geniuses,EG
18516,Evil Geniuses,2021-12-31,1017.52,1017.95,-0.86,1030.38,-41.76,983.16,982.74,+0.83,...,1609.67,181.93,-46.33,1820.31,1965.99,58.27,-2.36,39.0,Evil Geniuses,EG
18599,Evil Geniuses,2021-12-30,1017.64,1018.08,-0.87,1031.74,-42.04,983.04,982.62,+0.84,...,1609.67,181.93,-46.33,1820.31,1965.99,58.27,-2.36,39.0,Evil Geniuses,EG
18682,Evil Geniuses,2021-12-29,1017.77,1018.20,-0.88,1033.11,-42.33,982.92,982.50,+0.84,...,1609.67,181.93,-46.33,1820.31,1965.99,58.27,-2.36,39.0,Evil Geniuses,EG


In [97]:
best_2_df['team_elo_(k=32)_rating'].min()

827.58

In [98]:
best_2_df['opp_team_elo_(k=32)_rating'].min()

827.58

In [99]:
best_2_df[['team_elo_(k=32)_rating','opp_team_elo_(k=32)_rating']] = best_2_df[['team_elo_(k=32)_rating','opp_team_elo_(k=32)_rating']].fillna(800.0)

In [100]:
best_2_df['elo_diff'] = best_2_df['team_elo_(k=32)_rating'] - best_2_df['opp_team_elo_(k=32)_rating']

In [101]:
best_2_df.to_csv('best_2.csv',index=False)